# pythontutorial.net - Python Basics, Sections 12-15

In [1]:
# work inside a scratch folder so the file examples don't clutter the assignment folder
import os
from pathlib import Path

WORK = Path("tutorial_12_15_workspace").resolve()
WORK.mkdir(exist_ok=True)
os.chdir(WORK)
print("working in", os.getcwd())

working in /home/ksathwik/ece5831-2026/tutorial_12_15_workspace


## Section 12. Modules & Packages
### Modules

In [2]:
Path("pricing.py").write_text('''
def get_net_price(price, tax_rate, discount=0):
    return price * (1 + tax_rate) * (1 - discount)


def get_tax(price, tax_rate=0):
    return price * tax_rate
''')

import sys
sys.path.insert(0, str(WORK))

import pricing
print(pricing.get_net_price(price=100, tax_rate=0.01))

from pricing import get_net_price as calculate_net_price
print(calculate_net_price(100, 0.1, 0.05))

import pricing as selling_price
print(selling_price.get_tax(100, 0.1))

101.0
104.50000000000001
10.0


### Module search path

In [3]:
for path in sys.path[:6]:
    print(path)
print(pricing.__file__)

/home/ksathwik/ece5831-2026/tutorial_12_15_workspace
/home/ksathwik/.conda/envs/ece-5831-2026/lib/python312.zip
/home/ksathwik/.conda/envs/ece-5831-2026/lib/python3.12
/home/ksathwik/.conda/envs/ece-5831-2026/lib/python3.12/lib-dynload

/home/ksathwik/.conda/envs/ece-5831-2026/lib/python3.12/site-packages
/home/ksathwik/ece5831-2026/tutorial_12_15_workspace/pricing.py


### `__name__`

In [4]:
Path("billing.py").write_text('''
import pricing

def calculate_tax(price, tax):
    return price * tax

def print_billing_doc():
    print(f"billing module __name__ = {__name__}")
    print(f"Total with tax: {calculate_tax(100, 0.1)}")

if __name__ == "__main__":
    print_billing_doc()
''')

import billing
print("imported: billing.__name__ =", billing.__name__)

import subprocess
print(subprocess.run([sys.executable, "billing.py"], capture_output=True, text=True).stdout)

imported: billing.__name__ = billing
billing module __name__ = __main__
Total with tax: 10.0



### Packages

In [5]:
pkg = Path("sales")
pkg.mkdir(exist_ok=True)
(pkg / "__init__.py").write_text('''
# the package's public API
from .order import create_sales_order
from .delivery import create_delivery

TAX_RATE = 0.07
''')
(pkg / "order.py").write_text('def create_sales_order():\n    return "Create a sales order"\n')
(pkg / "delivery.py").write_text('def create_delivery():\n    return "Create a delivery"\n')

import sales
from sales.order import create_sales_order
print(create_sales_order())
print(sales.create_delivery())
print(sales.TAX_RATE)

Create a sales order
Create a delivery
0.07


### Private functions (`_name` and `__all__`)

In [6]:
Path("mail.py").write_text('''
__all__ = ["send"]

def send(email, message):
    return f"Sending '{message}' to {email}" + _attach_file("a.txt")

def _attach_file(filename):
    return f" (with {filename})"
''')

ns = {}
exec("from mail import *", ns)
print("send" in ns, "_attach_file" in ns)
print(ns["send"]("test@example.com", "Hello"))

True False
Sending 'Hello' to test@example.com (with a.txt)


## Section 13. Working with Files
### Write text files

In [7]:
lines = ["Readme", "How to write text files in Python"]
with open("readme.txt", "w") as f:
    f.write("\n".join(lines))

more_lines = ["", "Append text files", "The End"]
with open("readme.txt", "a") as f:
    f.write("\n".join(more_lines))

quote = "\u9759\u4ee5\u4fee\u8eab\uff0c\u4fed\u4ee5\u517b\u5fb7"
with open("quotes.txt", "w", encoding="utf8") as f:
    f.write(quote)
print("files written")

files written


### Read text files

In [8]:
with open("readme.txt") as f:
    print(f.read())
print("---")
with open("readme.txt") as f:
    lines = f.readlines()
print(lines)
print("---")
with open("readme.txt") as f:
    for line in f:
        print(line.strip())

with open("quotes.txt", encoding="utf8") as f:
    print(f.read())

Readme
How to write text files in Python
Append text files
The End
---
['Readme\n', 'How to write text files in Python\n', 'Append text files\n', 'The End']
---
Readme
How to write text files in Python
Append text files
The End
静以修身，俭以养德


### Create a new file (mode `x` fails if it already exists)

In [9]:
with open("new.txt", "x") as f:
    f.write("created with mode x")
try:
    with open("new.txt", "x") as f:
        pass
except FileExistsError as e:
    print("FileExistsError:", e)

FileExistsError: [Errno 17] File exists: 'new.txt'


### Check if a file exists

In [10]:
import os.path
print(os.path.exists("readme.txt"))
print(os.path.isfile("readme.txt"), os.path.isfile("sales"))
print(Path("missing.txt").exists())

True
True False
False


### Read a CSV file

In [11]:
import csv

Path("country.csv").write_text(
    "name,area,country_code2,country_code3\n"
    "Afghanistan,652090.00,AF,AFG\n"
    "Albania,28748.00,AL,ALB\n"
    "Algeria,2381741.00,DZ,DZA\n"
    "American Samoa,199.00,AS,ASM\n")

with open("country.csv", encoding="utf8") as f:
    csv_reader = csv.reader(f)
    next(csv_reader)  # skip header
    total = 0
    for row in csv_reader:
        print(row)
        total += float(row[1])
print(f"Total area: {total}")

with open("country.csv", encoding="utf8") as f:
    for line in csv.DictReader(f):
        print(line["name"], line["country_code2"])

['Afghanistan', '652090.00', 'AF', 'AFG']
['Albania', '28748.00', 'AL', 'ALB']
['Algeria', '2381741.00', 'DZ', 'DZA']
['American Samoa', '199.00', 'AS', 'ASM']
Total area: 3062778.0
Afghanistan AF
Albania AL
Algeria DZ
American Samoa AS


### Write a CSV file

In [12]:
header = ["name", "area", "country_code2", "country_code3"]
data = [["Albania", 28748, "AL", "ALB"],
        ["Algeria", 2381741, "DZ", "DZA"],
        ["American Samoa", 199, "AS", "ASM"]]

with open("countries.csv", "w", encoding="UTF8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(header)
    writer.writerows(data)

rows = [{"name": "Albania", "area": 28748, "country_code2": "AL", "country_code3": "ALB"}]
with open("countries_dict.csv", "w", encoding="UTF8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=header)
    writer.writeheader()
    writer.writerows(rows)

print(Path("countries.csv").read_text())
print(Path("countries_dict.csv").read_text())

name,area,country_code2,country_code3
Albania,28748,AL,ALB
Algeria,2381741,DZ,DZA
American Samoa,199,AS,ASM

name,area,country_code2,country_code3
Albania,28748,AL,ALB



### Rename a file

In [13]:
os.rename("readme.txt", "notes.txt")
print(os.path.exists("readme.txt"), os.path.exists("notes.txt"))
try:
    os.rename("readme.txt", "notes2.txt")
except FileNotFoundError as e:
    print("FileNotFoundError:", e)

False True
FileNotFoundError: [Errno 2] No such file or directory: 'readme.txt' -> 'notes2.txt'


### Delete a file

In [14]:
os.remove("new.txt")
print(os.path.exists("new.txt"))
try:
    os.remove("new.txt")
except FileNotFoundError as e:
    print("FileNotFoundError:", e)

False
FileNotFoundError: [Errno 2] No such file or directory: 'new.txt'


## Section 14. Working with Directories
### Directory operations

In [15]:
print("cwd:", os.getcwd())

os.mkdir("temp")
os.makedirs(os.path.join("temp", "a", "b"), exist_ok=True)
print(os.path.exists("temp"), os.path.isdir("temp"))

os.rename("temp", "temp2")
print(os.path.join("temp2", "a", "b"))

os.chdir("temp2")
print("cwd:", os.getcwd())
os.chdir("..")

import shutil
shutil.rmtree("temp2")
print(os.path.exists("temp2"))

cwd: /home/ksathwik/ece5831-2026/tutorial_12_15_workspace
True True
temp2/a/b
cwd: /home/ksathwik/ece5831-2026/tutorial_12_15_workspace/temp2
False


### List files in a directory

In [16]:
for root, dirs, files in os.walk("."):
    for file in files:
        if file.endswith(".py"):
            print(os.path.join(root, file))

print("---")
print(sorted(p.name for p in Path(".").glob("*.csv")))
print(sorted(p.as_posix() for p in Path(".").rglob("*.py")))

./billing.py
./mail.py
./pricing.py
./sales/order.py
./sales/delivery.py
./sales/__init__.py
---
['countries.csv', 'countries_dict.csv', 'country.csv']
['billing.py', 'mail.py', 'pricing.py', 'sales/__init__.py', 'sales/delivery.py', 'sales/order.py']


## Section 15. Third-party Packages
### PyPI and pip

In [17]:
import subprocess

def run(*args):
    out = subprocess.run([sys.executable, "-m", *args], capture_output=True, text=True)
    print(out.stdout or out.stderr)

run("pip", "--version")
run("pip", "show", "pandapower")

pip 26.2.1 from /home/ksathwik/.conda/envs/ece-5831-2026/lib/python3.12/site-packages/pip (python 3.12)



Name: pandapower
Version: 3.5.4
Summary: An easy to use open source tool for power system modeling, analysis and optimization with a high degree of automation.
Home-page: https://www.pandapower.org
Author: 
Author-email: Leon Thurner <leon.thurner@retoflow.de>, Alexander Scheidler <alexander.scheidler@iee.fraunhofer.de>, Mike Vogt <mike.vogt@iee.fraunhofer.de>
License: 
Location: /home/ksathwik/.conda/envs/ece-5831-2026/lib/python3.12/site-packages
Requires: colorama, deepdiff, geojson, networkx, numpy, packaging, pandas, pandera, scipy, tqdm, typing_extensions
Required-by: 



In [18]:
# pip list, filtered to a few packages installed in this environment
out = subprocess.run([sys.executable, "-m", "pip", "list"], capture_output=True, text=True).stdout
for line in out.splitlines():
    if line.split()[0].lower() in {"package", "pandapower", "pandas", "numpy", "ipykernel", "pip"} or line.startswith("---"):
        print(line)

Package                   Version
------------------------- ------------
ipykernel                 7.3.0
numpy                     2.4.6
pandapower                3.5.4
pandas                    2.3.3
pip                       26.2.1


In [19]:
import pandapower as pp

net = pp.create_empty_network()
b1 = pp.create_bus(net, vn_kv=20.0, name="Bus 1")
b2 = pp.create_bus(net, vn_kv=0.4, name="Bus 2")
b3 = pp.create_bus(net, vn_kv=0.4, name="Bus 3")
pp.create_ext_grid(net, bus=b1, vm_pu=1.02, name="Grid Connection")
pp.create_transformer(net, hv_bus=b1, lv_bus=b2, std_type="0.4 MVA 20/0.4 kV", name="Trafo")
pp.create_line(net, from_bus=b2, to_bus=b3, length_km=0.1, std_type="NAYY 4x50 SE", name="Line")
pp.create_load(net, bus=b3, p_mw=0.1, q_mvar=0.05, name="Load")
pp.runpp(net)
print(net.res_bus)

      vm_pu   va_degree      p_mw    q_mvar
0  1.020000    0.000000 -0.107265 -0.052675
1  1.008843 -150.760126  0.000000  0.000000
2  0.964431 -149.884141  0.100000  0.050000


### Virtual environments (venv)

In [20]:
import venv

venv.create("project_env", with_pip=False)
cfg = Path("project_env", "pyvenv.cfg")
print(cfg.read_text())
print("in a virtual env right now?", sys.prefix != sys.base_prefix)
print("sys.prefix:", sys.prefix)
shutil.rmtree("project_env")

home = /home/ksathwik/.conda/envs/ece-5831-2026/bin
include-system-site-packages = false
version = 3.12.14
executable = /home/ksathwik/.conda/envs/ece-5831-2026/bin/python3.12
command = /home/ksathwik/.conda/envs/ece-5831-2026/bin/python -m venv --copies --without-pip /home/ksathwik/ece5831-2026/tutorial_12_15_workspace/project_env

in a virtual env right now? False
sys.prefix: /home/ksathwik/.conda/envs/ece-5831-2026


### pipenv
`pipenv` combines pip + virtualenv and tracks dependencies in a `Pipfile`. Typical commands:
```
pip install pipenv
pipenv install requests
pipenv shell
pipenv run python app.py
```

In [21]:
pipenv_path = shutil.which("pipenv")
print("pipenv installed:", pipenv_path is not None)

# a Pipfile like the one `pipenv install requests` would generate
print('''[[source]]
url = "https://pypi.org/simple"
verify_ssl = true
name = "pypi"

[packages]
requests = "*"

[requires]
python_version = "3.12"''')

pipenv installed: False
[[source]]
url = "https://pypi.org/simple"
verify_ssl = true
name = "pypi"

[packages]
requests = "*"

[requires]
python_version = "3.12"


### Clean up

In [22]:
os.chdir(WORK.parent)
shutil.rmtree(WORK)
print("removed", WORK)

removed /home/ksathwik/ece5831-2026/tutorial_12_15_workspace
